In [ ]:
import os
import pandas as pd
import numpy as np

# Define the base directory path
dir_path = "/home/VECVNET/zzasom/jupyter/01_Developement_work/RG/Aging_Analysis"

# Explicit list of files from your screenshot
file_list = [
    "1161.09.09.2026.xls",
    "1159.09.09.2026.xls",
    "1158.09.09.2026.xls",
    "1157.09.09.2026.xls",
    "1156.09.09.2026.xls",
    "1155.09.09.2026.xls",
    "1153.09.09.2026.xls",
    "1152.09.09.2026.xls",
    "1141.09.09.2026.xls",
    "1110.09.09.2026.xls"
]

df_list = []

print("Starting strict data cleaning and extraction layout alignment...")

# Helper function to handle bad SAP numeric strings (e.g., "1,250.00-", " - ")
def clean_sap_numeric(val):
    val_str = str(val).strip()
    if not val_str or val_str in ['-', '', 'nan', 'NaN']:
        return 0.0
    
    # Remove thousand separators
    val_str = val_str.replace(',', '')
    
    # Handle trailing negative sign (SAP standard)
    if val_str.endswith('-'):
        val_str = '-' + val_str[:-1]
        
    try:
        return float(val_str)
    except ValueError:
        return np.nan

for file_name in file_list:
    full_path = os.path.join(dir_path, file_name)
    
    if os.path.exists(full_path):
        try:
            # Step 1: Open file with universal newlines handling to find header row location
            with open(full_path, 'r', encoding='utf-16', errors='ignore') as f:
                lines = f.readlines()
            
            # Find the true header line index (case-insensitive checking)
            header_idx = 0
            for idx, line in enumerate(lines):
                lower_line = line.lower()
                if 'plant' in lower_line or 'material' in lower_line or 'stoc' in lower_line:
                    header_idx = idx
                    break
            
            # Step 2: Read dataframe using python engine which is safer for broken rows
            df = pd.read_csv(
                full_path, 
                sep='\t', 
                encoding='utf-16', 
                skiprows=header_idx, 
                engine='python', 
                on_bad_lines='skip'
            )
            
            # Step 3: Column headers clean aur standardized mapping karna
            df.columns = df.columns.str.strip()
            
            # Truncated or alternative header variations mapping
            rename_dict = {}
            for col in df.columns:
                col_lower = col.lower()
                if any(x in col_lower for x in ['plant stoc', 'pl stock', 'pl.stock', 'plant stock', 'unrestricted']):
                    rename_dict[col] = 'Plant Stock'
                elif any(x in col_lower for x in ['tot.us.va', 'total value', 'tot value', 'valuated price']):
                    rename_dict[col] = 'Total Value'
                elif any(x in col_lower for x in ['moving pr', 'mvavgprice', 'moving price', 'price']):
                    rename_dict[col] = 'Moving Price'
                elif any(x in col_lower for x in ['material ci descr.', 'descr.', 'description', 'mat. desc.']):
                    rename_dict[col] = 'Description'
                    
            if rename_dict:
                df.rename(columns=rename_dict, inplace=True)
            
            # Step 4: Initial drop for blank structures
            df.dropna(how='all', inplace=True)
            
            # Step 5: Clean whitespace characters/newlines values inside cells before validation
            for col in df.columns:
                if df[col].dtype == object:
                    df[col] = df[col].astype(str).str.replace(r'\r|\n', ' ', regex=True).str.strip()

            # Master Row Clean Logic (SAP lines shift aur wrapping fix)
            # Sirf wahi rows bachaenge jinka Plant numeric character structure text match karega
            if 'Plant' in df.columns:
                df = df[df['Plant'].notna()]
                df = df[pd.to_numeric(df['Plant'], errors='coerce').notna()]
            
            if 'Material' in df.columns:
                df = df[df['Material'].notna()]
                df = df[df['Material'].astype(str).str.strip() != '']
            
            # Step 6: Parse numerical values strictly with our Custom SAP Function
            numeric_cols = ['Plant Stock', 'Total Value', 'Moving Price']
            for col in numeric_cols:
                if col in df.columns:
                    df[col] = df[col].apply(clean_sap_numeric)
            
            # Step 7: Back-calculate missing 'Plant Stock' if Total Value and Moving Price are present
            # Formula: Plant Stock = Total Value / Moving Price
            if 'Plant Stock' in df.columns and 'Total Value' in df.columns and 'Moving Price' in df.columns:
                # Fill missing or zeroed stock values only if valid price matrices are found
                mask_nan = (df['Plant Stock'].isna() | (df['Plant Stock'] == 0)) & df['Total Value'].notna() & (df['Moving Price'] > 0)
                df.loc[mask_nan, 'Plant Stock'] = df.loc[mask_nan, 'Total Value'] / df.loc[mask_nan, 'Moving Price']
                
                # Round to 2 decimal places to match standard SAP layout numbers
                df['Plant Stock'] = df['Plant Stock'].round(2)
            
            if len(df) > 0:
                # Format IDs nicely (Removing .0 formatting from raw floats strings)
                for col in ['Plant', 'Material']:
                    if col in df.columns:
                        df[col] = df[col].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
                
                df['source_file'] = file_name
                df_list.append(df)
                print(f"Successfully cleaned & aligned: {file_name} ({len(df)} perfect records)")
            else:
                print(f"Warning: No valid lines inside {file_name} post filtering pass.")
                
        except Exception as e:
            print(f"Error handling content mapping inside {file_name}: {str(e)}")
    else:
        print(f"Warning: File not found at {full_path}")

# Merge everything into a master file
if df_list:
    master_df = pd.concat(df_list, ignore_index=True, sort=False)
    
    # Final pass cleaning to remove empty layout columns
    master_df.dropna(how='all', axis=1, inplace=True)
    
    # Save clean dataset to Excel
    output_path = os.path.join(dir_path, "master_aging_analysis.xlsx")
    master_df.to_excel(output_path, index=False)
    
    print(f"\nSuccess! Completely clean Master file created with shape: {master_df.shape}")
    print(f"Saved directly to: {output_path}")
else:
    print("\nNo data collected. Please verify your data directory structure or files format.")

In [2]:
master_df = pd.read_excel("/home/VECVNET/zzasom/jupyter/01_Developement_work/RG/Aging_Analysis/master_aging_analysis.xlsx")

In [3]:
master_df1 = master_df[['Plant', 'Material','Plant Stock','SEP2026', 'AUG2026', 'JUL2026', 'JUN2026', 'MAY2026', 
    'APR2026', 'MAR2026', 'FEB2026', 'JAN2026', 'DEC2025', 'NOV2025', 
    'OCT2025', '> 1', '> 2', '> 3']]

In [4]:
import pandas as pd

# --- New Step: Remove leading zeros from Material column ---
if 'Material' in master_df1.columns:
    # Ensure it's a string, strip leading zeros, and clean blank spaces
    master_df1['Material'] = master_df1['Material'].astype(str).str.lstrip('0').str.strip()

# Your existing numeric formatting script block
columns_to_convert = [
    'Plant','Plant Stock','SEP2026', 'AUG2026', 'JUL2026', 'JUN2026', 'MAY2026', 
    'APR2026', 'MAR2026', 'FEB2026', 'JAN2026', 'DEC2025', 'NOV2025', 
    'OCT2025', '> 1', '> 2', '> 3'
]

for col in columns_to_convert:
    if col in master_df1.columns:
        # 1. Clean the text formatting if it's an object/string type column
        if master_df1[col].dtype == 'object':
            # Remove spaces, commas, and common currency symbols
            master_df1[col] = (
                master_df1[col]
                .astype(str)
                .str.replace(r'[^\d\.\-]', '', regex=True) # Removes everything except numbers, dots, and minus signs
                .str.strip()
            )
        
        # 2. Force conversion to numbers (invalid text safely becomes NaN)
        numeric_series = pd.to_numeric(master_df1[col], errors='coerce')
        
        # 3. Round decimals to whole numbers and cast to nullable integer
        master_df1[col] = numeric_series.round(0).astype('Int64')

/tmp/ipykernel_122954/4281875320.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  master_df1['Material'] = master_df1['Material'].astype(str).str.lstrip('0').str.strip()
/tmp/ipykernel_122954/4281875320.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  master_df1[col] = numeric_series.round(0).astype('Int64')
/tmp/ipykernel_122954/4281875320.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

In [5]:
df1 = pd.read_csv("/home/VECVNET/zzasom/jupyter/01_Developement_work/RG/Aging_Analysis/ECC_STOCK_20260910.CSV")
#df1.head(3)

In [ ]:
# Convert text/object column to float first to bypass text spaces, then fill blanks with 0
df1['Un.Stock'] = pd.to_numeric(df1['Un.Stock'], errors='coerce').round(0).astype("Int64")
# Use the capital "Int64" to allow missing/blank entries safely
#df1['column_name'] = pd.to_numeric(master_df['column_name'], errors='coerce').astype("Int64")
df1['Plant'] = df1['Plant'].replace([np.inf, -np.inf], np.nan)
# Now safely cast it to an integer
#df1['Plant'] = df1['Plant'].astype(int)
#df1[Plant] = df1[Plant].replace([np.inf, -np.inf], np.nan)
df1['Plant'] = df1['Plant'].fillna(0).astype(int)
clean_df = df1[['Plant', 'Material', 'Un.Stock','StorageLoc.']]

In [6]:
# Clean join columns
master_df1["Plant"] = pd.to_numeric(
    master_df1["Plant"], errors="coerce"
)

clean_df["Plant"] = pd.to_numeric(
    clean_df["Plant"], errors="coerce"
)

master_df1["Material"] = (
    master_df1["Material"].astype("string").str.strip()
)

clean_df["Material"] = (
    clean_df["Material"].astype("string").str.strip()
)

# Remove missing Plant / Material values
master_df1 = master_df1.dropna(
    subset=["Plant", "Material"]
)

clean_df = clean_df.dropna(
    subset=["Plant", "Material"]
)

# Make Plant same integer type
master_df1["Plant"] = master_df1["Plant"].astype(int)
clean_df["Plant"] = clean_df["Plant"].astype(int)

# Remove duplicate keys from clean_df
clean_df = clean_df.drop_duplicates(
    subset=["Plant", "Material"],
    keep="first"
)

# Inner join
final = master_df1.merge(
    clean_df,
    on=["Plant", "Material"],
    how="inner"
)

# Reset index
final.reset_index(
    drop=True,
    inplace=True
)

print("Final shape:", final.shape)
final = final.fillna(0)

/tmp/ipykernel_122954/3164088535.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  master_df1["Plant"] = pd.to_numeric(


NameError: name 'clean_df' is not defined

In [ ]:
final = final.rename(columns={
    'Plant Stock': 'Plant_stock',
    'Un.Stock': 'Saleable_stock'})

In [ ]:
final['Non_Saleable_stock'] = final['Plant_stock'] - final['Saleable_stock']

In [ ]:
df_filtered = final.copy()

In [ ]:
#df_filtered = final[final['StorageLoc.'] != 'GE04']
#df_filtered.shape

In [ ]:
import pandas as pd
import os


df_filtered = df_filtered.loc[:, ~df_filtered.columns.duplicated()]


month_columns = [
    'SEP2026', 'AUG2026', 'JUL2026', 'JUN2026', 'MAY2026', 
    'APR2026', 'MAR2026', 'FEB2026', 'JAN2026', 'DEC2025', 'NOV2025', 
    'OCT2025', '> 1', '> 2', '> 3'
]


aging_months_list = []

print("calculate Aging month")

for idx, row in df_filtered.iterrows():
    stock_to_reduce = row['Non_Saleable_stock']
    
    
    if isinstance(stock_to_reduce, pd.Series):
        stock_to_reduce = stock_to_reduce.iloc[0]
        
    if pd.isna(stock_to_reduce) or stock_to_reduce <= 0:
        aging_months_list.append("")
        continue
        
    stock_to_reduce = int(stock_to_reduce)
    affected_months = []
    
    # 
    for month in month_columns:
        if month in final.columns:
            current_month_val = final.at[idx, month]
            
            # 
            if isinstance(current_month_val, pd.Series):
                current_month_val = current_month_val.iloc[0]
                
            if pd.isna(current_month_val):
                current_month_val = 0
            else:
                current_month_val = int(current_month_val)
            
            if current_month_val > 0 and stock_to_reduce > 0:
                affected_months.append(month)
                
                if stock_to_reduce >= current_month_val:
                    stock_to_reduce -= current_month_val
                    final.at[idx, month] = 0
                else:
                    final.at[idx, month] = current_month_val - stock_to_reduce
                    stock_to_reduce = 0
                    
            if stock_to_reduce == 0:
                break
                
    aging_months_list.append(", ".join(affected_months))


df_filtered['Aging_month'] = aging_months_list
print("Done")



front_columns = ['Plant', 'Material', 'Plant_stock', 'Saleable_stock', 'Non_Saleable_stock'] + month_columns + ['Aging_month']

#final DataFrame 
ordered_front_cols = [col for col in front_columns if col in df_filtered.columns]

# 
remaining_cols = [col for col in df_filtered.columns if col not in front_columns]

#
final_columns_order = ordered_front_cols + remaining_cols

df_filtered = df_filtered[final_columns_order]

print("successfully!!!")